# 03 - Reversible at the maximum batch size

1. Find the largest batch (in sequences of 512 tokens) that completes a full training step, for the
   standard model and for the best reversible variant (doubling + binary search, granularity 8).
2. Train the reversible model for the same 50M tokens at that batch size.
   With a fixed token budget, a bigger batch means fewer optimizer steps, so the learning rate is scaled
   with sqrt(B/32) (capped at 3e-3) and warm-up is set to 5% of steps.

Peak memory here uses fine-grained sampling (after every Linear/LayerNorm/attention call and on each of
their gradients), which also captures the transient activations *inside* a layer.
On MPS, attention materialises the full (B, heads, T, T) score matrix and MPSGraph rejects tensors with more
than INT_MAX elements (first hit at B=1,536), so attention is split over the batch above 2^30 scores;
the maximum batch is then set by memory, not by that backend limit.

In [1]:
# --- setup: works on Google Colab (CUDA) and locally (Apple MPS / CUDA / CPU) ---
import os, sys
# MPS: make the allocator raise a clean OOM at the GPU's recommended working-set size
# instead of spilling into swap (must be set before torch is imported).
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "1.0")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.9")
if "google.colab" in sys.modules and not os.path.exists("revllm.py"):
    !git clone -q -b session13 https://github.com/chapanduk/ERAV5.git /content/ERAV5
    %cd /content/ERAV5/session13
    !pip -q install tiktoken pyarrow
if not os.path.exists("data/train.bin"):
    !python prepare_data.py
import json, torch, revllm
from revllm import ModelConfig, TrainConfig, train, pick_device
DEVICE = pick_device()
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name() if DEVICE == "cuda" else "")
TRAIN_BIN, VAL_BIN = "data/train.bin", "data/val.bin"
GiB = 2**30

torch 2.15.0 | device mps | 


In [2]:
BEST = json.load(open("results/02_best_variant.json"))["best"]
print("reversible variant:", BEST)
SEARCH_JSON = "results/03_max_batch_search.json"
if os.path.exists(SEARCH_JSON):  # reuse a completed search (it takes ~10 min); delete the file to re-run
    search = json.load(open(SEARCH_JSON))
    print("loaded cached search from", SEARCH_JSON)
else:
    search = {}
    for arch in ["standard", BEST]:
        maxB, trials = revllm.find_max_batch(ModelConfig(arch=arch), DEVICE, start=32, limit=8192)
        search[arch] = {"max_batch": maxB, "trials": trials}
    json.dump(search, open(SEARCH_JSON, "w"), indent=1)
for arch, s in search.items():
    print(f"=== {arch}: max batch (1-2 step probe) = {s['max_batch']} sequences = {s['max_batch']*512:,} tokens/step")
    for t in s["trials"]:
        print(f"   B={t['B']:5d} " + (f"ok   peak {t['peak_mem_bytes']/GiB:6.2f} GiB  {t['tokens_per_sec']:8.0f} tok/s" if t["ok"]
              else "FAIL " + t["error"][:90]))

reversible variant: momentum
loaded cached search from results/03_max_batch_search.json
=== standard: max batch (1-2 step probe) = 216 sequences = 110,592 tokens/step
   B=   32 ok   peak   7.33 GiB     14071 tok/s
   B=   64 ok   peak  14.13 GiB     15052 tok/s
   B=  128 ok   peak  27.73 GiB     15180 tok/s
   B=  256 FAIL MPS backend out of memory (MPS allocated: 51.41 GiB, other allocations: 96.70 MiB, max all
   B=  192 ok   peak  41.33 GiB      7179 tok/s
   B=  224 FAIL MPS backend out of memory (MPS allocated: 50.87 GiB, other allocations: 228.78 MiB, max al
   B=  208 ok   peak  44.73 GiB      6924 tok/s
   B=  216 ok   peak  47.00 GiB      6938 tok/s
=== momentum: max batch (1-2 step probe) = 1528 sequences = 782,336 tokens/step
   B=   32 ok   peak   1.42 GiB     20293 tok/s
   B=   64 ok   peak   2.24 GiB     19214 tok/s
   B=  128 ok   peak   3.88 GiB     14522 tok/s
   B=  256 ok   peak   7.17 GiB     14333 tok/s
   B=  512 ok   peak  13.73 GiB     13300 tok/s
   B= 1024 

In [3]:
import math, gc
# The probe max only has to survive 1-2 steps on a fresh allocator. Sustained training fragments the MPS
# allocator, so at ~99% of the limit it can still OOM a few steps in. Start at the probe max and step down
# by 64 sequences on OOM; the first batch that trains for the full 50M tokens is the sustained max batch.
B = search[BEST]["max_batch"]
attempts = []
while True:
    lr = min(3e-3, 1e-3 * math.sqrt(B / 32))
    print(f"training {BEST} at B={B} ({B*512:,} tok/step, {50_000_000//(B*512)} steps), lr={lr:.2e}")
    try:
        res, model = train(ModelConfig(arch=BEST),
                           TrainConfig(name=f"03_reversible_{BEST}_maxB{B}", batch_size=B, lr=lr,
                                       warmup_frac=0.05, log_every=1),
                           TRAIN_BIN, VAL_BIN, DEVICE)
        attempts.append({"B": B, "ok": True}); break
    except (RuntimeError, torch.OutOfMemoryError) as e:
        if not revllm._is_oom(e): raise
        revllm.MEM = None
        msg = str(e).splitlines()[0][:160]
        attempts.append({"B": B, "ok": False, "error": msg}); print("  OOM:", msg)
        gc.collect(); torch.mps.empty_cache() if DEVICE == "mps" else torch.cuda.empty_cache()
        B -= 64
json.dump(attempts, open("results/03_sustained_batch_attempts.json", "w"), indent=1)
print("sustained max batch:", B, attempts)

training momentum at B=1528 (782,336 tok/step, 63 steps), lr=3.00e-03


[03_reversible_momentum_maxB1528] arch=momentum params=40.77M (non-emb 21.25M) device=mps B=1528 T=512 tokens/step=782336 steps=63


[03_reversible_momentum_maxB1528] step     1/63 tok   0.78M loss 10.9135 gnorm 3.39   10.8k tok/s peak 39.65GiB


[03_reversible_momentum_maxB1528] step     2/63 tok   1.56M loss 10.3135 gnorm 2.00   10.8k tok/s peak 39.95GiB


[03_reversible_momentum_maxB1528] step     3/63 tok   2.35M loss 9.6080 gnorm 1.70    9.7k tok/s peak 39.95GiB


[03_reversible_momentum_maxB1528] step     4/63 tok   3.13M loss 8.7872 gnorm 1.64   10.0k tok/s peak 39.95GiB


... (67 outputs left out)

[03_reversible_momentum_maxB1528] step    63/63 tok  49.29M loss 7.3476 gnorm 0.09   10.8k tok/s peak 39.95GiB


[03_reversible_momentum_maxB1528]   val_loss 7.3679


[03_reversible_momentum_maxB1528] DONE final_train_loss=7.3577 final_val_loss=7.3679 tok/s=9973 peak=39.95GiB time=82.2min


sustained max batch: 1528 [{'B': 1528, 'ok': True}]
